# RAW to ACES Render Time Comparison

Measure and compare conversion times for converting RAW images to ACES format using rawtoaces.

## Import Required Libraries

Import necessary libraries for file handling, timing operations, and displaying results.

In [8]:
import os
import glob
import subprocess
import time
from pathlib import Path

import numpy as np

## Load Sample Images from Dataset

Locate the first 10 RAW image files from `dataset/temp/raw` directory.

In [9]:
# Define the dataset path using absolute path
notebooks_dir = Path.cwd()
raw_dir = (notebooks_dir.parent / "dataset" / "temp" / "raw").resolve()

# Get all RAW files (CR2, NEF, ARW, RAF, DNG)
raw_extensions = ["*.CR2", "*.NEF", "*.ARW", "*.RAF", "*.DNG"]
raw_files = []
for ext in raw_extensions:
    raw_files.extend(sorted(raw_dir.glob(ext)))

# Select first 10 files
sample_files = raw_files[:10]

print(f"Raw directory: {raw_dir}")
print(f"Total RAW files found: {len(raw_files)}")
print(f"Selected {len(sample_files)} files for conversion:")
for i, f in enumerate(sample_files, 1):
    print(f"  {i}. {f.name}")

Raw directory: /run/media/mikkelkp/Projects/Projects/med8_project/LuminaScale/dataset/temp/raw
Total RAW files found: 1000
Selected 10 files for conversion:
  1. 0_0.CR2
  2. 0_1.CR2
  3. 0_2.CR2
  4. 0_3.CR2
  5. 0_4.CR2
  6. 0_5.CR2
  7. 0_6.CR2
  8. 1000_0.CR2
  9. 1000_1.CR2
  10. 1000_2.CR2


## Convert RAW Images to ACES Format

Convert each RAW image to ACES format using the rawtoaces command.

In [10]:
# Setup output directory using absolute path
notebook_dir = Path.cwd()
output_dir = notebook_dir.parent / "dataset" / "temp" / "aces_converted"
output_dir.mkdir(parents=True, exist_ok=True)

# Also create with absolute path for rawtoaces
output_dir_abs = output_dir.resolve()

# Storage for timing data
conversion_times = []
results = []

print(f"Notebook directory: {notebook_dir}")
print(f"Output directory: {output_dir_abs}\n")
print("Starting conversions...")
print("-" * 60)

Notebook directory: /run/media/mikkelkp/Projects/Projects/med8_project/LuminaScale/notebooks
Output directory: /run/media/mikkelkp/Projects/Projects/med8_project/LuminaScale/dataset/temp/aces_converted

Starting conversions...
------------------------------------------------------------


In [11]:
# Convert each file and record timing, then display statistics
for idx, raw_file in enumerate(sample_files, 1):
    filename = raw_file.name
    output_path = output_dir_abs / f"{raw_file.stem}.exr"
    
    # Measure conversion time
    start_time = time.time()
    
    try:
        # Run rawtoaces command with proper flags (matching bash script)
        cmd = [
            "rawtoaces",
            "--data-dir", "/usr/local/share/rawtoaces/data",
            "--output-dir", str(output_dir_abs),
            "--create-dirs",
            "--overwrite",
            str(raw_file.resolve())
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        
        elapsed_time = time.time() - start_time
        
        if result.returncode == 0:
            status = "Success"
        else:
            status = "Failed"
        
        conversion_times.append(elapsed_time)
        results.append({
            "Image": filename,
            "Time (seconds)": elapsed_time,
            "Status": status,
            "Output": output_path.name
        })
        
    except subprocess.TimeoutExpired:
        conversion_times.append(None)
        results.append({
            "Image": filename,
            "Time (seconds)": "Timeout",
            "Status": "Timeout",
            "Output": None
        })
    except Exception as e:
        conversion_times.append(None)
        results.append({
            "Image": filename,
            "Time (seconds)": "Error",
            "Status": "Error",
            "Output": None
        })

# Calculate and display statistics for successful conversions
valid_times = [t for t in conversion_times if isinstance(t, (int, float))]

if valid_times:
    total_time = sum(valid_times)
    avg_time = total_time / len(valid_times)
    min_time = min(valid_times)
    max_time = max(valid_times)
    
    print(f"\nConversion Time Statistics:")
    print(f"  Total time: {total_time:.2f} seconds")
    print(f"  Average time per image: {avg_time:.2f} seconds")
    print(f"  Minimum time: {min_time:.2f} seconds")
    print(f"  Maximum time: {max_time:.2f} seconds")
    print(f"  Successfully converted: {len(valid_times)}/{len(sample_files)}")
else:
    print("\nNo successful conversions to report.")


Conversion Time Statistics:
  Total time: 11.29 seconds
  Average time per image: 1.13 seconds
  Minimum time: 0.98 seconds
  Maximum time: 1.23 seconds
  Successfully converted: 10/10


## ACES2065-1 to sRGB Conversion with OpenImageIO

Convert ACES EXR images to 8-bit sRGB format using OpenImageIO's color space conversion.


In [12]:
import numpy as np
import torch
import OpenImageIO as oiio
import imageio
import psutil
import threading

# Force CPU device (disable GPU)
device = "cpu"

# ACES2065-1 to sRGB conversion matrices (using standard cinema pipeline)
ACES_TO_XYZ = torch.tensor([
    [0.6380, 0.2646, 0.0000],
    [0.2126, 0.7874, 0.0000],
    [0.0000, 0.0000, 1.0888]
], dtype=torch.float32, device=device)

XYZ_TO_SRGB = torch.tensor([
    [3.2406, -1.5372, -0.4986],
    [-0.9689, 1.8758, 0.0415],
    [0.0557, -0.2040, 1.0570]
], dtype=torch.float32, device=device)

def load_aces_and_convert_srgb_gpu(aces_path: Path | str) -> np.ndarray:
    """Load ACES2065-1 EXR and convert to sRGB using CPU PyTorch."""
    # Load EXR using OIIO
    inp = oiio.ImageInput.open(str(aces_path))
    if not inp:
        raise RuntimeError(f"Failed to open {aces_path}")
    
    image = inp.read_image()
    inp.close()
    
    # Handle metadata if available
    h, w = inp.spec().height, inp.spec().width
    c = inp.spec().nchannels
    
    # Convert to torch tensor on CPU
    img_tensor = torch.from_numpy(image).to(device).float()
    img_tensor = img_tensor.reshape(h, w, c)
    
    # Extract RGB channels (drop alpha if present)
    if c >= 3:
        rgb = img_tensor[:, :, :3]
    else:
        rgb = img_tensor
    
    # Reshape for matrix multiplication: [H*W, 3]
    h_t, w_t = rgb.shape[:2]
    rgb_flat = rgb.reshape(-1, 3)
    
    # Apply color conversion: ACES2065-1 -> XYZ -> sRGB
    xyz_flat = torch.matmul(rgb_flat, ACES_TO_XYZ.T)
    srgb_flat = torch.matmul(xyz_flat, XYZ_TO_SRGB.T)
    
    # Reshape back to image
    srgb = srgb_flat.reshape(h_t, w_t, 3)
    
    # Apply sRGB gamma correction
    def srgb_gamma(x):
        return torch.where(
            x <= 0.0031308,
            12.92 * x,
            1.055 * torch.pow(x, 1.0 / 2.4) - 0.055
        )
    
    srgb = srgb_gamma(srgb)
    
    # Clamp and convert back to numpy
    srgb = torch.clamp(srgb, 0.0, 1.0)
    srgb_np = srgb.cpu().numpy().astype(np.float32)
    
    return srgb_np

class CPUMonitor:
    """Monitor CPU usage during conversion."""
    def __init__(self):
        self.cpu_samples = []
        self.monitoring = False
        self.thread = None
    
    def _monitor(self):
        """Continuously sample CPU usage."""
        while self.monitoring:
            # Get per-core CPU percentages
            self.cpu_samples.append({
                'total': psutil.cpu_percent(interval=0.01),
                'per_core': psutil.cpu_percent(interval=0.01, percpu=True)
            })
    
    def start(self):
        """Start monitoring in background thread."""
        self.cpu_samples = []
        self.monitoring = True
        self.thread = threading.Thread(target=self._monitor, daemon=True)
        self.thread.start()
    
    def stop(self):
        """Stop monitoring."""
        self.monitoring = False
        if self.thread:
            self.thread.join(timeout=1)
    
    def get_stats(self):
        """Get CPU usage statistics."""
        if not self.cpu_samples:
            return None
        totals = [s['total'] for s in self.cpu_samples]
        return {
            'avg': np.mean(totals),
            'max': np.max(totals),
            'min': np.min(totals),
            'std': np.std(totals)
        }

# Setup directories
aces_src_dir = (Path.cwd().parent / "dataset" / "temp" / "aces").resolve()
aces_output_dir = (Path.cwd().parent / "dataset" / "temp" / "aces_converted").resolve()
aces_output_dir.mkdir(parents=True, exist_ok=True)

# Get all ACES EXR files
aces_files_list = sorted(list(aces_src_dir.glob("*_aces.exr")))

print(f"ACES source directory: {aces_src_dir}")
print(f"ACES output directory: {aces_output_dir}")
print(f"Device: {device}")
print(f"Total ACES images found: {len(aces_files_list)}")
print(f"CPU cores available: {psutil.cpu_count()}")
print("-" * 60)

# Batch convert and measure timing & CPU usage
aces_conversion_times = []
aces_results = []
cpu_monitor = CPUMonitor()

for idx, aces_path in enumerate(aces_files_list, 1):
    filename = aces_path.name
    output_path = aces_output_dir / f"{aces_path.stem}.png"
    
    start_time = time.time()
    cpu_monitor.start()
    
    try:
        # CPU conversion
        srgb_image = load_aces_and_convert_srgb_gpu(aces_path)
        
        cpu_monitor.stop()
        cpu_stats = cpu_monitor.get_stats()
        
        # Convert to 8-bit and save
        srgb_8bit = (np.clip(srgb_image, 0, 1) * 255).astype(np.uint8)
        imageio.imwrite(output_path, srgb_8bit)
        
        elapsed_time = time.time() - start_time
        status = "Success"
        aces_conversion_times.append(elapsed_time)
        aces_results.append({
            "Image": filename,
            "Time (seconds)": elapsed_time,
            "CPU avg %": round(cpu_stats['avg'], 1) if cpu_stats else "N/A",
            "CPU max %": round(cpu_stats['max'], 1) if cpu_stats else "N/A",
            "Status": status
        })
        
    except Exception as e:
        cpu_monitor.stop()
        elapsed_time = time.time() - start_time
        print(f"Error processing {filename}: {e}")
        aces_conversion_times.append(None)
        aces_results.append({
            "Image": filename,
            "Time (seconds)": "Error",
            "CPU avg %": "N/A",
            "CPU max %": "N/A",
            "Status": "Failed"
        })

print("-" * 60)

# Display statistics
aces_valid_times = [t for t in aces_conversion_times if isinstance(t, (int, float))]

if aces_valid_times:
    aces_total_time = sum(aces_valid_times)
    aces_avg_time = aces_total_time / len(aces_valid_times)
    aces_min_time = min(aces_valid_times)
    aces_max_time = max(aces_valid_times)
    
    # Calculate average CPU usage across all conversions
    valid_results = [r for r in aces_results if r["Status"] == "Success"]
    cpu_avgs = [r["CPU avg %"] for r in valid_results if isinstance(r["CPU avg %"], (int, float))]
    cpu_maxs = [r["CPU max %"] for r in valid_results if isinstance(r["CPU max %"], (int, float))]
    
    print(f"\nACES to sRGB Conversion Time Statistics (CPU PyTorch):")
    print(f"  Total time: {aces_total_time:.2f} seconds")
    print(f"  Average time per image: {aces_avg_time:.2f} seconds")
    print(f"  Minimum time: {aces_min_time:.2f} seconds")
    print(f"  Maximum time: {aces_max_time:.2f} seconds")
    print(f"  Successfully converted: {len(aces_valid_times)}/{len(aces_files_list)}")
    
    if cpu_avgs:
        print(f"\nCPU Usage Statistics:")
        print(f"  Average CPU usage: {np.mean(cpu_avgs):.1f}%")
        print(f"  Max CPU usage: {np.max(cpu_maxs):.1f}%")
        print(f"  CPU cores: {psutil.cpu_count()}")
else:
    print("\nNo successful ACES conversions to report.")

ACES source directory: /run/media/mikkelkp/Projects/Projects/med8_project/LuminaScale/dataset/temp/aces
ACES output directory: /run/media/mikkelkp/Projects/Projects/med8_project/LuminaScale/dataset/temp/aces_converted
Device: cpu
Total ACES images found: 10
CPU cores available: 16
------------------------------------------------------------
------------------------------------------------------------

ACES to sRGB Conversion Time Statistics (CPU PyTorch):
  Total time: 16.07 seconds
  Average time per image: 1.61 seconds
  Minimum time: 1.47 seconds
  Maximum time: 1.75 seconds
  Successfully converted: 10/10

CPU Usage Statistics:
  Average CPU usage: 61.6%
  Max CPU usage: 100.0%
  CPU cores: 16


In [13]:
import subprocess

def get_gpu_memory_usage():
    """Get GPU memory usage using nvidia-smi."""
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,nounits,noheader"],
            capture_output=True, text=True, timeout=5
        )
        if result.returncode == 0:
            used, total = map(int, result.stdout.strip().split(','))
            return {'used': used, 'total': total, 'percent': (used / total) * 100}
    except:
        pass
    return None

def run_conversion_benchmark(device_type: str, num_samples: int = None):
    """Run conversion benchmark on specified device."""
    if device_type not in ["cpu", "cuda"]:
        raise ValueError("device_type must be 'cpu' or 'cuda'")
    
    if device_type == "cuda" and not torch.cuda.is_available():
        print(f"CUDA not available, skipping GPU benchmark")
        return None
    
    # Prepare device-specific tensors
    device = device_type
    aces_to_xyz = torch.tensor([
        [0.6380, 0.2646, 0.0000],
        [0.2126, 0.7874, 0.0000],
        [0.0000, 0.0000, 1.0888]
    ], dtype=torch.float32, device=device)
    
    xyz_to_srgb = torch.tensor([
        [3.2406, -1.5372, -0.4986],
        [-0.9689, 1.8758, 0.0415],
        [0.0557, -0.2040, 1.0570]
    ], dtype=torch.float32, device=device)
    
    def convert_on_device(aces_path):
        """Convert ACES to sRGB on specified device."""
        # Load EXR
        inp = oiio.ImageInput.open(str(aces_path))
        if not inp:
            raise RuntimeError(f"Failed to open {aces_path}")
        
        image = inp.read_image()
        inp.close()
        
        h, w = inp.spec().height, inp.spec().width
        c = inp.spec().nchannels
        
        # Convert to tensor on specified device
        img_tensor = torch.from_numpy(image).to(device).float()
        img_tensor = img_tensor.reshape(h, w, c)
        
        if c >= 3:
            rgb = img_tensor[:, :, :3]
        else:
            rgb = img_tensor
        
        h_t, w_t = rgb.shape[:2]
        rgb_flat = rgb.reshape(-1, 3)
        
        # Color conversion
        xyz_flat = torch.matmul(rgb_flat, aces_to_xyz.T)
        srgb_flat = torch.matmul(xyz_flat, xyz_to_srgb.T)
        srgb = srgb_flat.reshape(h_t, w_t, 3)
        
        # Gamma correction
        def srgb_gamma(x):
            return torch.where(
                x <= 0.0031308,
                12.92 * x,
                1.055 * torch.pow(x, 1.0 / 2.4) - 0.055
            )
        
        srgb = srgb_gamma(srgb)
        srgb = torch.clamp(srgb, 0.0, 1.0)
        
        return srgb.cpu().numpy().astype(np.float32)
    
    # Get files to process
    files_to_process = aces_files_list if num_samples is None else aces_files_list[:num_samples]
    
    conversion_times = []
    cpu_usage = []
    gpu_memory = []
    results = []
    cpu_monitor = CPUMonitor()
    
    print(f"\n{'='*60}")
    print(f"Running conversion on {device_type.upper()}")
    print(f"{'='*60}")
    print(f"Files to process: {len(files_to_process)}")
    
    for idx, aces_path in enumerate(files_to_process, 1):
        filename = aces_path.name
        output_path = aces_output_dir / f"{filename.replace('_aces', f'_{device_type}')}"
        
        start_time = time.time()
        cpu_monitor.start()
        gpu_mem_start = get_gpu_memory_usage() if device_type == "cuda" else None
        
        try:
            srgb_image = convert_on_device(aces_path)
            
            cpu_monitor.stop()
            gpu_mem_end = get_gpu_memory_usage() if device_type == "cuda" else None
            
            # Save result
            srgb_8bit = (np.clip(srgb_image, 0, 1) * 255).astype(np.uint8)
            imageio.imwrite(output_path, srgb_8bit)
            
            elapsed_time = time.time() - start_time
            cpu_stats = cpu_monitor.get_stats()
            
            conversion_times.append(elapsed_time)
            cpu_usage.append(cpu_stats['avg'] if cpu_stats else 0)
            
            result_dict = {
                "Image": filename,
                "Time (s)": round(elapsed_time, 3),
                "CPU avg %": round(cpu_stats['avg'], 1) if cpu_stats else "N/A",
                "Status": "Success"
            }
            
            if device_type == "cuda":
                gpu_mem_used = gpu_mem_end['percent'] - gpu_mem_start['percent'] if gpu_mem_end and gpu_mem_start else 0
                result_dict["GPU mem %"] = round(gpu_mem_end['percent'], 1) if gpu_mem_end else "N/A"
                gpu_memory.append(gpu_mem_end['percent'] if gpu_mem_end else 0)
            
            results.append(result_dict)
            print(f"  [{idx}/{len(files_to_process)}] {filename}: {elapsed_time:.3f}s", end="")
            if cpu_stats:
                print(f" (CPU: {cpu_stats['avg']:.1f}%)", end="")
            if device_type == "cuda" and gpu_mem_end:
                print(f" (GPU mem: {gpu_mem_end['percent']:.1f}%)", end="")
            print()
            
        except Exception as e:
            cpu_monitor.stop()
            print(f"  [{idx}/{len(files_to_process)}] {filename}: ERROR - {e}")
            conversion_times.append(None)
            results.append({
                "Image": filename,
                "Time (s)": "Error",
                "CPU avg %": "N/A",
                "GPU mem %": "N/A" if device_type == "cuda" else None,
                "Status": "Failed"
            })
    
    # Compute statistics
    valid_times = [t for t in conversion_times if isinstance(t, (int, float))]
    
    if not valid_times:
        print(f"\nNo successful conversions on {device_type.upper()}")
        return None
    
    stats = {
        'device': device_type,
        'total_time': sum(valid_times),
        'avg_time': np.mean(valid_times),
        'min_time': np.min(valid_times),
        'max_time': np.max(valid_times),
        'std_time': np.std(valid_times),
        'avg_cpu_usage': np.mean(cpu_usage) if cpu_usage else 0,
        'max_cpu_usage': np.max(cpu_usage) if cpu_usage else 0,
        'successful': len(valid_times),
        'total': len(files_to_process),
        'gpu_memory': np.mean(gpu_memory) if gpu_memory else 0,
        'results': results
    }
    
    # Print summary
    print(f"\n{device_type.upper()} Summary:")
    print(f"  Total time: {stats['total_time']:.2f}s")
    print(f"  Average time/image: {stats['avg_time']:.3f}s")
    print(f"  Min/Max time: {stats['min_time']:.3f}s / {stats['max_time']:.3f}s")
    print(f"  Avg CPU usage: {stats['avg_cpu_usage']:.1f}%")
    if device_type == "cuda":
        print(f"  Avg GPU memory: {stats['gpu_memory']:.1f}%")
    print(f"  Success rate: {stats['successful']}/{stats['total']}")
    
    return stats

# Run benchmarks on both devices
print(f"\n{'#'*60}")
print(f"CPU vs GPU CONVERSION BENCHMARK")
print(f"{'#'*60}")
print(f"Available ACES images: {len(aces_files_list)}")
print(f"CPU cores: {psutil.cpu_count()}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

cpu_stats = run_conversion_benchmark("cpu", num_samples=len(aces_files_list))

if torch.cuda.is_available():
    gpu_stats = run_conversion_benchmark("cuda", num_samples=len(aces_files_list))
    
    # Comparison
    print(f"\n{'='*60}")
    print(f"PERFORMANCE COMPARISON")
    print(f"{'='*60}")
    if cpu_stats and gpu_stats:
        speedup = cpu_stats['avg_time'] / gpu_stats['avg_time']
        print(f"GPU speedup vs CPU: {speedup:.2f}x faster")
        print(f"CPU avg time/image: {cpu_stats['avg_time']:.3f}s")
        print(f"GPU avg time/image: {gpu_stats['avg_time']:.3f}s")
        print(f"CPU total time: {cpu_stats['total_time']:.2f}s")
        print(f"GPU total time: {gpu_stats['total_time']:.2f}s")
else:
    print(f"\nGPU not available - CPU benchmark only")



############################################################
CPU vs GPU CONVERSION BENCHMARK
############################################################
Available ACES images: 10
CPU cores: 16
CUDA available: True
GPU: NVIDIA GeForce RTX 3080

Running conversion on CPU
Files to process: 10
  [1/10] 0_0_aces.exr: 1.028s (CPU: 59.7%)
  [2/10] 0_1_aces.exr: 0.806s (CPU: 59.0%)
  [3/10] 0_2_aces.exr: 0.815s (CPU: 62.7%)
  [4/10] 0_3_aces.exr: 0.788s (CPU: 61.6%)
  [5/10] 0_4_aces.exr: 0.794s (CPU: 60.4%)
  [6/10] 0_5_aces.exr: 0.785s (CPU: 61.2%)
  [7/10] 0_6_aces.exr: 0.794s (CPU: 60.9%)
  [8/10] 1000_0_aces.exr: 0.745s (CPU: 63.4%)
  [9/10] 1000_1_aces.exr: 0.754s (CPU: 64.0%)
  [10/10] 1000_2_aces.exr: 0.745s (CPU: 62.1%)

CPU Summary:
  Total time: 8.05s
  Average time/image: 0.805s
  Min/Max time: 0.745s / 1.028s
  Avg CPU usage: 61.5%
  Success rate: 10/10

Running conversion on CUDA
Files to process: 10
  [1/10] 0_0_aces.exr: 0.360s (CPU: 34.1%) (GPU mem: 24.1%)
  [2/10] 0_1_aces.

## CPU vs GPU Performance Comparison

Compare ACES to sRGB conversion performance on both CPU and GPU devices with resource monitoring.
